### Import et Configuration

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration des axes cibles
AXES_CIBLES = ["Bd_Richard_Lenoir", "Quai_Conti", "Sts_Peres"]
FICHIER_CSV = 'data/raw/comptages.csv'

print(" Configuration chargée")
print(f" Fichier cible : {FICHIER_CSV}")
print(f" Axes analysés : {AXES_CIBLES}")

 Configuration chargée
 Fichier cible : data/raw/comptages.csv
 Axes analysés : ['Bd_Richard_Lenoir', 'Quai_Conti', 'Sts_Peres']


### Chargement

In [1]:
import pandas as pd

# 1. Configuration
FICHIER_CSV = 'data/raw/comptages.csv'
AXES_CIBLES = ["Bd_Richard_Lenoir", "Quai_Conti", "Sts_Peres"]

print("Chargement des données...")

# 2. Chargement direct (sans chunks)
df = pd.read_csv(FICHIER_CSV, sep=';')

# 3. Filtrage pour ne garder que nos 3 axes
df = df[df['libelle'].isin(AXES_CIBLES)].copy()

print(f" Chargement terminé !")
print(f" Nombre total de lignes : {len(df)}")
print(f" Nombre de colonnes : {df.shape[1]}")

Chargement des données...
 Chargement terminé !
 Nombre total de lignes : 69824
 Nombre de colonnes : 11


### Aperçu des données avec .head()


In [2]:
print("--- APERÇU DES DONNÉES (.head()) ---")
display(df.head())

--- APERÇU DES DONNÉES (.head()) ---


,iu_ac,libelle,t_1h,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre
0,1314,Bd_Richard_Lenoir,2026-08-31T23:00:00+00:00,180.0,0.90333,Fluide,722,Bd_Richard_Lenoir-St_Sebastien,721,Bd_Richard_Lenoir-Pelee-Moufle,Ouvert
1,1317,Bd_Richard_Lenoir,2026-08-31T23:00:00+00:00,83.0,0.25056,Fluide,721,Bd_Richard_Lenoir-Pelee-Moufle,722,Bd_Richard_Lenoir-St_Sebastien,Ouvert
2,1319,Bd_Richard_Lenoir,2026-08-31T23:00:00+00:00,83.0,0.26889,Fluide,722,Bd_Richard_Lenoir-St_Sebastien,723,Bd_Richard_Lenoir-Chemin_Vert,Ouvert
3,69,Quai_Conti,2026-08-31T23:00:00+00:00,607.0,4.23000,Fluide,61,Pt_Neuf_Gauche,62,Quai_Conti-Guenegaud,Ouvert
4,70,Quai_Conti,2026-08-31T23:00:00+00:00,607.0,4.23000,Fluide,62,Quai_Conti-Guenegaud,63,Institut,Ouvert


### 1. Format date en datetime et heure de paris, identifiant en texte(string) et etat en catégory
   Le but ici est de convertir la colonne 't_1h' en datetime avec le fuseau horaire de Paris


In [3]:


# 1. Conversion de la colonne 't_1h' en datetime avec le fuseau horaire de Paris
# On suppose d'abord que les dates sont en UTC, puis on les convertit à l'heure de Paris
df['t_1h'] = pd.to_datetime(df['t_1h'], utc=True).dt.tz_convert('Europe/Paris')

# 2. Conversion des identifiants en texte (string)
colonnes_identifiants = ['iu_ac', 'iu_nd_amont', 'iu_nd_aval']
for col in colonnes_identifiants:
    df[col] = df[col].astype(str)

# 3. Conversion des colonnes d'état en catégorie
colonnes_etat = ['etat_trafic', 'etat_barre']
for col in colonnes_etat:
    df[col] = df[col].astype('category')

# Vérification des types de données après modification
print("Types de données après transformation :")
print(df.dtypes)

Types de données après transformation :
iu_ac                                        str
libelle                                      str
t_1h                datetime64[us, Europe/Paris]
q                                        float64
k                                        float64
etat_trafic                             category
iu_nd_amont                                  str
libelle_nd_amont                             str
iu_nd_aval                                   str
libelle_nd_aval                              str
etat_barre                              category
dtype: object


### 2. Verification de doublons à l'aide de la bonne clé et si il y'en a on vas devoir les supprimer


In [ ]:
# 1. Vérification des doublons globaux (toutes les colonnes strictement identiques)
nb_doublons_global = df.duplicated().sum()
print(f"Nombre de lignes strictement identiques (toutes colonnes) : {nb_doublons_global}")

# 2. Définition de la bonne clé unique
# Une mesure de trafic est unique par identifiant de tronçon et par timestamp
cle_unique = ['iu_ac', 't_1h']

# 3. Vérification des doublons sur cette clé spécifique
nb_doublons_cle = df.duplicated(subset=cle_unique).sum()
print(f"Nombre de lignes en double sur la clé {cle_unique} : {nb_doublons_cle}")


Nombre de lignes strictement identiques (toutes colonnes) : 0
Nombre de lignes en double sur la clé ['iu_ac', 't_1h'] : 0


In [15]:
def groupes_identiques(df, col):
    """Associe à chaque capteur un numéro de groupe : même numéro = série identique."""
    # Note : j'utilise 't_1h' car c'est ta colonne datetime. 
    # Si tu as créé une colonne 't_utc', remplace 't_1h' par 't_utc' ici.
    large = df.pivot_table(index="t_1h", columns="iu_ac", values=col)
    signature = large.round(3).fillna(-1).T.apply(lambda s: hash(tuple(s)), axis=1)
    return signature.map({h: i for i, h in enumerate(signature.unique())})

groupe_q = groupes_identiques(df, "q")
groupe_k = groupes_identiques(df, "k")
print(f"Debit q      : {len(groupe_q)} capteurs, {groupe_q.nunique()} series differentes")
print(f"Occupation k : {len(groupe_k)} capteurs, {groupe_k.nunique()} series differentes")

# --- Partie optionnelle visible en bas de ta capture ---
# Identifier le plus grand groupe de capteurs au débit identique
plus_grand = groupe_q.value_counts().index[0]
capteurs_identiques = groupe_q[groupe_q == plus_grand].index.tolist()

print(f"\nLe plus grand groupe de capteurs au debit identique (groupe {plus_grand}) contient {len(capteurs_identiques)} capteurs :")
print(capteurs_identiques)

Debit q      : 16 capteurs, 16 series differentes
Occupation k : 16 capteurs, 16 series differentes

Le plus grand groupe de capteurs au debit identique (groupe 0) contient 1 capteurs :
['1314']


Puisque mon DataFrame df est filtré sur mes 3 axes, le code compte le nombre de capteurs (iu_ac) présents sur ces 3 axes.
Vu que j'ai obtenu 16 capteurs, 16 séries différentes, cela veut dire que même au niveau microscopique des capteurs, il n'y a aucun doublon caché. 
 S'il n'y a pas de doublons, le plus grand "paquet" de capteurs identiques ne peut logiquement avoir une taille que de 1 qui est ici le capteur '1314'

### 3. L'objectif est d'identifier les colonnes qui apportent la même information (redondance) et de ne garder que la plus utile

Identifier les redondances potentielles
Dans mon DataFrame, on peut suspecter comme redondance :
iu_nd_amont / libelle_nd_amont : l'ID et le nom du nœud amont (même info, deux formats)
iu_nd_aval / libelle_nd_aval : idem pour le nœud aval
etat_trafic / etat_barre : deux états différents mais potentiellement liés


In [16]:
# --- TEST 1 : Redondance entre ID et libellé des nœuds ---
# Si chaque ID correspond à un seul libellé (et vice-versa), les colonnes sont redondantes

test_amont = df.groupby('iu_nd_amont')['libelle_nd_amont'].nunique()
test_aval = df.groupby('iu_nd_aval')['libelle_nd_aval'].nunique()

print("--- Test de redondance ID <-> Libellé ---")
print(f"nœuds amont : {len(test_amont)} IDs uniques -> {test_amont.max()} libellés max par ID")
print(f"nœuds aval  : {len(test_aval)} IDs uniques -> {test_aval.max()} libellés max par ID")

if test_amont.max() == 1 and test_aval.max() == 1:
    print("\n CONSTAT : Relation 1-1 parfaite. Les libellés sont redondants avec les IDs.")
    print("   -> On peut supprimer 'libelle_nd_amont' et 'libelle_nd_aval'")
    print("   -> Justification : chaque ID correspond à un seul libellé. Garder les deux alourdit le DataFrame sans apport d'information.")
else:
    print("\n CONSTAT : Relation non 1-1. On conserve les deux colonnes.")

--- Test de redondance ID <-> Libellé ---
nœuds amont : 12 IDs uniques -> 1 libellés max par ID
nœuds aval  : 12 IDs uniques -> 1 libellés max par ID

 CONSTAT : Relation 1-1 parfaite. Les libellés sont redondants avec les IDs.
   -> On peut supprimer 'libelle_nd_amont' et 'libelle_nd_aval'
   -> Justification : chaque ID correspond à un seul libellé. Garder les deux alourdit le DataFrame sans apport d'information.


In [17]:
# --- TEST 2 : Redondance entre etat_trafic et etat_barre ---
# On calcule un tableau de contingence pour voir si les états sont liés

contingence = pd.crosstab(df['etat_trafic'], df['etat_barre'])
print("\n--- Tableau de contingence : etat_trafic vs etat_barre ---")
print(contingence)

# Calcul du taux de co-occurrence
total = len(df)
print(f"\nTotal d'observations : {total}")
print("\nInterprétation :")
print("- Si 'Invalide' (etat_barre) correspond toujours à 'Inconnu' (etat_trafic) -> redondance partielle")
print("- Si 'Ouvert' correspond à plusieurs états de trafic -> pas de redondance, on garde les deux")


--- Tableau de contingence : etat_trafic vs etat_barre ---
etat_barre   Invalide  Ouvert
etat_trafic                  
Bloqué              0      41
Fluide              0   65908
Inconnu          1861     136
Pré-saturé          0    1496
Saturé              0     382

Total d'observations : 69824

Interprétation :
- Si 'Invalide' (etat_barre) correspond toujours à 'Inconnu' (etat_trafic) -> redondance partielle
- Si 'Ouvert' correspond à plusieurs états de trafic -> pas de redondance, on garde les deux


In [18]:
# --- TEST 3 : Redondance entre libelle et iu_ac ---
# Est-ce que chaque capteur (iu_ac) appartient à un seul axe (libelle) ?

test_capteur_axe = df.groupby('iu_ac')['libelle'].nunique()
print("\n--- Test : un capteur = un seul axe ? ---")
print(f"Nombre de capteurs : {len(test_capteur_axe)}")
print(f"Nombre max d'axes par capteur : {test_capteur_axe.max()}")

if test_capteur_axe.max() == 1:
    print(" CONSTAT : Chaque capteur appartient à un seul axe.")
    print("   -> 'libelle' est redondant avec 'iu_ac' (on peut retrouver l'axe via l'ID)")
    print("   -> MAIS on garde 'libelle' car c'est plus lisible pour l'analyse humaine")


--- Test : un capteur = un seul axe ? ---
Nombre de capteurs : 16
Nombre max d'axes par capteur : 1
 CONSTAT : Chaque capteur appartient à un seul axe.
   -> 'libelle' est redondant avec 'iu_ac' (on peut retrouver l'axe via l'ID)
   -> MAIS on garde 'libelle' car c'est plus lisible pour l'analyse humaine


In [19]:
# Suppression des colonnes redondantes identifiées
colonnes_a_supprimer = []

# Si le test 1 a montré une relation 1-1, on supprime les libellés de nœuds
if test_amont.max() == 1 and test_aval.max() == 1:
    colonnes_a_supprimer.extend(['libelle_nd_amont', 'libelle_nd_aval'])
    print(" Suppression de : libelle_nd_amont, libelle_nd_aval (redondants avec les IDs)")

# On ne supprime PAS etat_trafic ni etat_barre car ils apportent des infos différentes
# On ne supprime PAS libelle car c'est plus lisible que iu_ac

if colonnes_a_supprimer:
    df = df.drop(columns=colonnes_a_supprimer)
    print(f"\n DataFrame après nettoyage : {df.shape[1]} colonnes (au lieu de {df.shape[1] + len(colonnes_a_supprimer)})")
else:
    print("\n Aucune colonne redondante détectée. On conserve tout.")

print("\n--- Colonnes finales ---")
print(df.columns.tolist())

 Suppression de : libelle_nd_amont, libelle_nd_aval (redondants avec les IDs)

 DataFrame après nettoyage : 9 colonnes (au lieu de 11)

--- Colonnes finales ---
['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'iu_nd_amont', 'iu_nd_aval', 'etat_barre']


Nous avons testé trois hypothèses de redondance :
1. ID vs Libellé des nœuds : Vérification d'une relation 1-1 entre `iu_nd_amont`/`libelle_nd_amont` et `iu_nd_aval`/`libelle_nd_aval`.
2. États croisés : Analyse de la corrélation entre `etat_trafic` et `etat_barre` via un tableau de contingence.
3. Capteur vs Axe : Vérification si `iu_ac` détermine uniquement `libelle`.
D'après les résultats
-Les libellés de nœuds sont redondants avec les IDs (relation 1-1). Nous les supprimons pour alléger le DataFrame.
- Les colonnes `etat_trafic` et `etat_barre` ne sont pas redondantes : elles capturent des dimensions différentes (état du flux vs état physique de la voirie). Nous les conservons toutes les deux.
- `libelle` est techniquement redondant avec `iu_ac`, mais nous le conservons pour la lisibilité de l'analyse.

DataFrame final : [9] colonnes conservées, toutes porteuses d'information distincte.

### 4. Verification des valeurs manquantes : que veulent dire *Barré* et *Invalide* sur vos axes ? Quelle est la longueur de vos trous ? Choisissez votre méthode à partir de ces chiffres.

In [20]:
# Fonction pour analyser le comportement de q et k selon l'état
def analyser_etat(df, colonne_etat):
    print(f"--- Analyse de la colonne : {colonne_etat} ---")
    
    # On crée un tableau récapitulatif
    resume = df.groupby(colonne_etat).agg(
        nb_lignes=(colonne_etat, 'count'),
        q_moyen=('q', 'mean'),
        q_min=('q', 'min'),
        q_max=('q', 'max'),
        q_nan_pct=('q', lambda x: x.isna().sum() / len(x) * 100),
        k_moyen=('k', 'mean')
    ).round(2)
    
    display(resume)
    print("\n")

# Analyse sur l'ensemble du DataFrame filtré (tes 3 axes)
analyser_etat(df, 'etat_trafic')
analyser_etat(df, 'etat_barre')

--- Analyse de la colonne : etat_trafic ---


,nb_lignes,q_moyen,q_min,q_max,q_nan_pct,k_moyen
etat_trafic,,,,,,
Bloqué,41,752.07,138.0,942.0,0.0,55.34
Fluide,65908,338.77,0.0,1521.0,0.0,3.55
Inconnu,1997,415.91,27.0,1437.2,0.0,3.53
Pré-saturé,1496,750.77,32.0,1563.0,0.0,20.51
Saturé,382,798.85,121.0,1496.0,0.0,37.04




--- Analyse de la colonne : etat_barre ---


,nb_lignes,q_moyen,q_min,q_max,q_nan_pct,k_moyen
etat_barre,,,,,,
Invalide,1861,424.73,49.5,1437.2,0.0,3.44
Ouvert,67963,350.59,0.0,1563.0,0.0,4.15


 1. Signification des états "Invalide" et "Inconnu au lieu de barré pour nos axes choisi" : Vérification par les données

Contrairement à l'hypothèse initiale, l'analyse empirique révèle que **les états "Inconnu au lieu de barré" et "Invalide" ne correspondent pas à des données manquantes**.

**Interprétation :**
- **Aucun NaN** : La colonne `q` ne contient aucune valeur manquante, quel que soit l'état déclaré. Les capteurs ont bien envoyé des mesures.
- **"Inconnu" et "Invalide" ≈ "Fluide"** : Les valeurs de débit et d'occupation pour ces états sont très proches de l'état "Fluide". Cela suggère qu'il ne s'agit pas de pannes de capteurs, mais de **périodes où l'algorithme de classification du trafic n'arrive pas à attribuer un état clair** (probablement en heures creuses, quand le trafic est très faible et difficile à classifier).
- **"Bloqué"** : Débit élevé (752 véh/h) avec occupation très forte (55%). Cohérent avec un embouteillage : les voitures sont présentes mais avancent très lentement.

**Conclusion :**
Les états "Inconnu" et "Invalide" ne représentent pas des données manquantes, mais des **incertitudes de classification**. Les valeurs de `q` et `k` sont bien présentes et exploitables. Le traitement de ces états doit donc se concentrer sur la **fiabilité de la classification** plutôt que sur l'imputation de valeurs manquantes.

2. Longueurs des trous

In [21]:

# 1. Créer un masque combinant les 3 types de "trous"
# - etat_trafic est Inconnu
# - etat_barre est Invalide
# - Débit quasi-nul (q <= 5 véh/h, seuil négligeable)
df['est_trou'] = (
    df['etat_trafic'].isin(['Inconnu']) |
    df['etat_barre'].isin(['Invalide']) |
    (df['q'] <= 5)
)

# 2. Trier par axe et par heure pour garantir la continuité temporelle
df_sorted = df.sort_values(by=['libelle', 't_1h']).copy()

# 3. Identifier les blocs consécutifs de trous
# On crée un nouvel ID de groupe à chaque fois que l'état change
df_sorted['changement'] = df_sorted['est_trou'] != df_sorted['est_trou'].shift(1)
df_sorted['id_groupe'] = df_sorted['changement'].cumsum()

# 4. Isoler uniquement les lignes qui sont des trous
trous = df_sorted[df_sorted['est_trou']].copy()

if len(trous) == 0:
    print("Aucun trou détecté dans les données.")
else:
    # 5. Calculer la durée de chaque groupe de trous
    # Comme les données sont horaires, la durée = (heure de fin - heure de début) + 1 heure
    resume_trous = trous.groupby(['libelle', 'id_groupe']).agg(
        debut=('t_1h', 'min'),
        fin=('t_1h', 'max'),
        duree_heures=('t_1h', lambda x: int((x.max() - x.min()).total_seconds() / 3600) + 1),
        nb_lignes=('est_trou', 'count'),
        type_dominant=('etat_trafic', lambda x: x.mode()[0] if not x.mode().empty else 'q_faible')
    ).reset_index()

    # Trier pour voir les trous les plus longs en premier pour chaque axe
    resume_trous = resume_trous.sort_values(by=['libelle', 'duree_heures'], ascending=[True, False])

    print("--- Détail des trous de données (les 15 plus longs) ---")
    display(resume_trous.head(15))
    
    # Statistiques globales
    duree_moyenne = resume_trous['duree_heures'].mean()
    duree_mediane = resume_trous['duree_heures'].median()
    duree_max = resume_trous['duree_heures'].max()
    nb_trous_courts = (resume_trous['duree_heures'] <= 3).sum()
    nb_trous_longs = (resume_trous['duree_heures'] > 3).sum()
    
    print(f"\n--- Statistiques globales ---")
    print(f"Nombre total de trous identifiés : {len(resume_trous)}")
    print(f"Durée moyenne d'un trou : {duree_moyenne:.1f} heures")
    print(f"Durée médiane d'un trou : {duree_mediane:.1f} heures")
    print(f"Durée maximale d'un trou : {duree_max} heures")
    print(f"\nRépartition :")
    print(f"  - Trous courts (<= 3h) : {nb_trous_courts} ({nb_trous_courts/len(resume_trous)*100:.1f}%)")
    print(f"  - Trous longs (> 3h) : {nb_trous_longs} ({nb_trous_longs/len(resume_trous)*100:.1f}%)")

--- Détail des trous de données (les 15 plus longs) ---


,libelle,id_groupe,debut,fin,duree_heures,nb_lignes,type_dominant
286,Bd_Richard_Lenoir,574,2026-04-14 01:00:00+02:00,2026-04-14 11:00:00+02:00,11,80,Inconnu
306,Bd_Richard_Lenoir,614,2026-06-03 13:00:00+02:00,2026-06-03 15:00:00+02:00,3,21,Inconnu
311,Bd_Richard_Lenoir,624,2026-06-07 13:00:00+02:00,2026-06-07 15:00:00+02:00,3,30,Inconnu
15,Bd_Richard_Lenoir,32,2026-03-19 20:00:00+01:00,2026-03-19 21:00:00+01:00,2,7,Inconnu
30,Bd_Richard_Lenoir,62,2026-03-23 12:00:00+01:00,2026-03-23 13:00:00+01:00,2,3,Inconnu
36,Bd_Richard_Lenoir,74,2026-03-23 16:00:00+01:00,2026-03-23 17:00:00+01:00,2,7,Inconnu
39,Bd_Richard_Lenoir,80,2026-03-23 18:00:00+01:00,2026-03-23 19:00:00+01:00,2,4,Inconnu
42,Bd_Richard_Lenoir,86,2026-03-23 22:00:00+01:00,2026-03-23 23:00:00+01:00,2,4,Inconnu
43,Bd_Richard_Lenoir,88,2026-03-23 23:00:00+01:00,2026-03-24 00:00:00+01:00,2,8,Inconnu
51,Bd_Richard_Lenoir,104,2026-03-27 13:00:00+01:00,2026-03-27 14:00:00+01:00,2,9,Inconnu



--- Statistiques globales ---
Nombre total de trous identifiés : 487
Durée moyenne d'un trou : 1.3 heures
Durée médiane d'un trou : 1.0 heures
Durée maximale d'un trou : 11 heures

Répartition :
  - Trous courts (<= 3h) : 484 (99.4%)
  - Trous longs (> 3h) : 3 (0.6%)


Nous avons identifié les périodes discontinues en combinant trois critères : état "Inconnu", état "Invalide", ou débit quasi-nul (q ≤ 5 véh/h). 

**Résultats chiffrés :**
- **Nombre total de périodes discontinues** : 487 sur l'ensemble des 3 axes.
- **Durée médiane** : 1.0 heure (la moitié des interruptions ne dure qu'une seule heure, typiquement la nuit).
- **Durée maximale** : 11 heures.
- **Répartition** : 99.4 % des trous durent 3 heures ou moins. Seuls 3 trous (0.6 %) dépassent cette durée.

**Justification de la méthode choisie :**
Ces chiffres dictent une stratégie de traitement nuancée et rigoureuse :
1. **Pour les trous courts (≤ 3h, soit 99.4 % des cas)** : Nous appliquons une **interpolation linéaire** (`method='linear'`). Comme l'interruption est très brève, estimer la valeur manquante en reliant les points valides avant et après le trou est statistiquement fiable et préserve la dynamique réelle du trafic sans créer de discontinuité artificielle.
2. **Pour les trous longs (> 3h, soit 0.6 % des cas)** : Nous choisissons de **ne pas interpoler** ces valeurs (elles restent en `NaN`). Sur une durée de 11 heures, la variation du trafic est trop importante pour être devinée par une simple droite. Imputer ces données introduirait un biais artificiel (inventer du trafic). Il est scientifiquement plus honnête de laisser ces valeurs manquantes, d'autant plus qu'elles sont trop rares pour impacter les agrégations globales (moyennes, médianes).

In [22]:

print(" Début du traitement des valeurs manquantes...")

# 1. Identifier les lignes problématiques
masque_trou = (
    df['etat_trafic'].isin(['Inconnu']) |
    df['etat_barre'].isin(['Invalide']) |
    (df['q'] <= 5)
)

# 2. Mettre q et k à NaN pour ces lignes
df.loc[masque_trou, 'q'] = np.nan
df.loc[masque_trou, 'k'] = np.nan

print(f"Nombre de valeurs mises à NaN : {masque_trou.sum()}")

# 3. Appliquer l'interpolation linéaire avec limite de 3 heures
# Le paramètre limit=3 empêche d'interpoler les trous de plus de 3 heures
df['q'] = df.groupby('libelle')['q'].transform(
    lambda x: x.interpolate(method='linear', limit=3, limit_direction='both')
)
df['k'] = df.groupby('libelle')['k'].transform(
    lambda x: x.interpolate(method='linear', limit=3, limit_direction='both')
)

# 4. Vérification finale
nb_nan_final_q = df['q'].isna().sum()
nb_nan_final_k = df['k'].isna().sum()
nb_nan_initial = masque_trou.sum()
nb_interpole = nb_nan_initial - nb_nan_final_q

print(f"\n Traitement terminé.")
print(f"Valeurs manquantes initiales : {nb_nan_initial}")
print(f"Valeurs interpolées (trous ≤ 3h) : {nb_interpole}")
print(f"Valeurs manquantes restantes (trous > 3h) : {nb_nan_final_q}")
print(f"Taux d'interpolation : {nb_interpole/nb_nan_initial*100:.1f}%")

 Début du traitement des valeurs manquantes...
Nombre de valeurs mises à NaN : 1999

 Traitement terminé.
Valeurs manquantes initiales : 1999
Valeurs interpolées (trous ≤ 3h) : 1596
Valeurs manquantes restantes (trous > 3h) : 403
Taux d'interpolation : 79.8%


Nous avons utilisé l'interpolation linéaire plutôt que temporelle car nos données sont collectées à pas horaire régulier. Dans ce cas, les deux méthodes produisent des résultats identiques, mais l'interpolation linéaire est plus simple à implémenter dans un contexte groupé

In [27]:
import os

# 1. Créer le dossier data/processed s'il n'existe pas
os.makedirs('data/processed', exist_ok=True)

# 2. Sauvegarder le DataFrame nettoyé
# On utilise Parquet (plus léger et rapide que CSV) ou CSV selon tes préférences
df.to_csv('data/processed/comptages_propres.csv', index=False, sep=';')

print("Données propres sauvegardées dans data/processed/comptages_propres.csv")
print(f"Nombre de lignes : {len(df)}")
print(f"Nombre de colonnes : {df.shape[1]}")

Données propres sauvegardées dans data/processed/comptages_propres.csv
Nombre de lignes : 69824
Nombre de colonnes : 10
